## Libraries

In [1]:
import os

os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"

In [36]:
import numpy as np 

import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

## Dataset

In [37]:
# Transform to turn into tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307),(0.3081,)) # mean y std
])

# Load the dataset and normalize it
full_train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

# Load the test dataset
full_test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=False,
    transform=transform
)

## Subsets

In [38]:
train_indices = torch.randperm(len(full_train_dataset))[:6000]
test_indices = torch.randperm(len(full_test_dataset))[:1000]

train_subset = Subset(full_train_dataset, train_indices)
test_subset = Subset(full_test_dataset, test_indices)

In [39]:
# DataLoaders
train_loader = DataLoader(
    dataset=train_subset,
    batch_size=64,
    num_workers=4,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_subset,
    batch_size=64,
    num_workers=4,
    shuffle=True,
    drop_last=False
)

## Train

In [40]:
def createTheMNISTNet():

    class mnistNet(nn.Module):
        def __init__(self):
            super().__init__()

            self.input = nn.Linear(784,64)

            self.fc1 = nn.Linear(64,32)
            self.fc2 = nn.Linear(32,32)

            self.output = nn.Linear(32,10)

        def forward(self,x):
            x = x.view(x.shape[0], -1)
            x = F.relu( self.input(x) )
            x = F.relu( self.fc1(x) )
            x = F.relu( self.fc2(x) )
            return self.output(x)

    net = mnistNet()

    lossfun = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(net.parameters(), lr=.01)

    return net, lossfun, optimizer

In [41]:
def function2trainTheModel():

    numepochs = 50

    net, lossfun, optimizer = createTheMNISTNet()

    losses = torch.zeros(numepochs)
    trainAcc = []
    testAcc = []

    for epochi in range(numepochs):

        batchAcc = []
        batchLoss = []
        for X, y in train_loader:

            yHat = net(X)
            loss = lossfun(yHat,y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batchLoss.append(loss.item())

            matches = torch.argmax(yHat, axis=1) == y
            matchesNumeric = matches.float()
            accuracyPct = 100*torch.mean(matchesNumeric)
            batchAcc.append( accuracyPct )

        trainAcc.append( np.mean(batchAcc) )

        losses[epochi] = np.mean(batchLoss)

        X,y = next(iter(test_loader))
        with torch.no_grad():
            yHat = net(X)

        testAcc.append( 100*torch.mean((torch.argmax(yHat, axis=1)==y).float()) )

        print(f'Epoch: {epochi} \f trainAcc: {trainAcc[epochi]} \f testAcc: {testAcc[epochi]}')

    return trainAcc, testAcc, losses, net

In [ ]:
trainAcc, testAcc, losses, net = function2trainTheModel()

fig, ax = plt.subplots(1,2, figsize=(16,5))

ax[0].plot(losses)
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_ylim([0,3])
ax[0].set_title('Model loss')

ax[1].plot(trainAcc, label='Train')
ax[1].plot(testAcc, label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_ylim([10,100])
ax[1].set_title(f'Final model test accuracy: {testAcc[-1]:.2f}%')
ax[1].legend()

plt.show()

Epoch: 0  trainAcc: 13.978494644165039  testAcc: 17.1875
Epoch: 1  trainAcc: 27.68817138671875  testAcc: 50.0
Epoch: 2  trainAcc: 45.967742919921875  testAcc: 50.0
Epoch: 3  trainAcc: 51.461692810058594  testAcc: 48.4375
Epoch: 4  trainAcc: 59.61021423339844  testAcc: 73.4375
Epoch: 5  trainAcc: 73.60551452636719  testAcc: 78.125
Epoch: 6  trainAcc: 79.30107879638672  testAcc: 87.5
Epoch: 7  trainAcc: 82.51007843017578  testAcc: 71.875
Epoch: 8  trainAcc: 84.87903594970703  testAcc: 89.0625
Epoch: 9  trainAcc: 86.7271499633789  testAcc: 89.0625
Epoch: 10  trainAcc: 87.43279266357422  testAcc: 92.1875
Epoch: 11  trainAcc: 88.42405700683594  testAcc: 85.9375
Epoch: 12  trainAcc: 89.29771423339844  testAcc: 85.9375
Epoch: 13  trainAcc: 89.56653594970703  testAcc: 90.625
Epoch: 14  trainAcc: 90.0369644165039  testAcc: 90.625
Epoch: 15  trainAcc: 90.57460021972656  testAcc: 93.75
Epoch: 16  trainAcc: 90.82661437988281  testAcc: 90.625
Epoch: 17  trainAcc: 